ASK 2: Chatbot for FAQs
● Collect FAQs related to a topic or product (questions and their answers).
● Preprocess the text using NLP libraries like NLTK or SpaCy (tokenize, clean, etc.).
● Match user questions with the most similar FAQ using techniques like cosine similarity or intent
matching.
● Display the best matching answer as a chatbot response.
● Optional: Create a simple chat UI for user interaction.

In [ ]:
# ✅ Rule-Based Chatbot

import re

def chatbot_response(user_input):
    text = user_input.lower()

    if re.search(r"\bhello\b|\bhi\b|\bhey\b", text):
        return "Hello! I'm your AI assistant. How can I help you today?"
    elif re.search(r"\bhow are you\b", text):
        return "I'm just a bunch of code, but I'm doing great! How about you?"
    elif re.search(r"\bweather\b|\btemperature\b", text):
        return "You can check real-time weather using an API, but I’d say it's always sunny inside Colab!"
    elif re.search(r"\bname\b", text):
        return "You can call me AlphaBot!"
    elif re.search(r"\bbye\b|\bexit\b|\bquit\b", text):
        return "Goodbye! Have a great day!"
    else:
        return "I'm not sure about that. Could you rephrase your question?"

print("🤖 Chatbot: Hello! Type 'bye' to exit.")
while True:
    user_input = input("You: ")
    if user_input.lower() in ['bye', 'exit', 'quit']:
        print("🤖 Chatbot:", chatbot_response(user_input))
        break
    print("🤖 Chatbot:", chatbot_response(user_input))

🤖 Chatbot: Hello! Type 'bye' to exit.
You: Hi
🤖 Chatbot: Hello! I'm your AI assistant. How can I help you today?
You: What's today's weather 
🤖 Chatbot: You can check real-time weather using an API, but I’d say it's always sunny inside Colab!
You: Bye
🤖 Chatbot: Goodbye! Have a great day!


In [ ]:


!pip install -q nltk scikit-learn sentence-transformers

# 1) Sample FAQ file creation (if you don't have one)
sample_csv = """
question,answer
"What is your return policy?","You can return items within 30 days with receipt."
"How do I reset my password?","Use the 'Forgot password' link on login and follow instructions."
"What shipping options are available?","We provide standard and express shipping. Costs vary with region."
"How can I contact support?","Email support@example.com or call +1-555-123-456."
"Do you offer bulk discounts?","Yes — contact sales@example.com for quotes on bulk orders."
"""
with open('faqs.csv','w',encoding='utf-8') as f:
    f.write(sample_csv.strip())

# 2) Load and preprocess
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk, re
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Download the missing resource
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

df = pd.read_csv('faqs.csv')
df['question'] = df['question'].astype(str)
df['answer'] = df['answer'].astype(str)

# simple text cleanup
def clean_text(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    tokens = nltk.word_tokenize(s)
    tokens = [t for t in tokens if t not in stop_words]
    return " ".join(tokens)

df['q_clean'] = df['question'].apply(clean_text)

# 3) TF-IDF vectorizer
vectorizer = TfidfVectorizer(ngram_range=(1,2)).fit(df['q_clean'])
tfidf_matrix = vectorizer.transform(df['q_clean'])

# 4) Chat function using cosine similarity
def faq_answer(user_question, top_k=1, threshold=0.2):
    uc = clean_text(user_question)
    vec = vectorizer.transform([uc])
    sims = cosine_similarity(vec, tfidf_matrix).flatten()
    best_idx = sims.argmax()
    best_score = sims[best_idx]
    if best_score < threshold:
        return None, best_score
    return df.iloc[best_idx]['answer'], best_score

# 5) Interactive loop
print("FAQ Chatbot ready. Type 'exit' to quit.")
while True:
    q = input("You: ").strip()
    if q.lower() in ('exit','quit'):
        print("Bot: Goodbye!")
        break
    ans, score = faq_answer(q)
    if ans:
        print(f"Bot (score {score:.2f}):", ans)
    else:
        print("Bot: Sorry, I couldn't find an answer. Try rephrasing or ask support@...")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


FAQ Chatbot ready. Type 'exit' to quit.
You: Hii
Bot: Sorry, I couldn't find an answer. Try rephrasing or ask support@...
You: How do I reset my password?
Bot (score 1.00): Use the 'Forgot password' link on login and follow instructions.
You: Exit 
Bot: Goodbye!
